# Lichen Ultra Validation

This notebook validates the standalone `lichen` blockwise hidden-memory env construction.

It keeps `window_size = 2` fixed for all tests, so every block contains two raw noisy segments.


## Notebook Plan

The intended progression mirrors the earlier env notebooks:

0. Frame-mapping micro-check for `Q_b` versus `F_b`
1. Test 1: one-qubit free evolution, `L=8`, all gates `I`
2. Test 2: one-qubit all-`X` case
3. Test 3: two-qubit free shared noise
4. Test 4: two-qubit alternating-`CNOT` cycle
5. Test 5: observable-level free vs DD comparison from `|+><+|`
6. Test 6: medium 3-qubit Clifford example
7. Test 7: big 8-qubit sampled blockwise check

Tests 1-6 use `window_size = 2` and `num_shots = 100`.
Test 7 also uses `window_size = 2`, but uses a smaller batch because the exact blockwise `ultra` path is still materially more expensive than the `max` segmentwise path.




In [1]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    src_dir = parent / "src"
    if (src_dir / "lichen").is_dir():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the local src/lichen package")

import time
import numpy as np

from lichen import (
    CircuitDescription,
    CircuitLayer,
    HiddenMemorySample,
    SharedQuasiStaticModel,
    SparseExportConfig,
    build_blockwise_hidden_memory_process,
    build_fixed_window_blocks,
    sample_blockwise_hidden_memory_processes,
)
from lichen.pauli import pauli_string_to_matrix



## Shared Helpers

This section holds lightweight helper constructors for the `ultra` blockwise env tests.

The important convention is that block probabilities are computed in the toggling frame, while sampled simulator faults are inserted in the physical frame after block-end conjugation.


In [2]:
from collections import Counter

WINDOW_SIZE = 2
NUM_SHOTS = 100
NUM_SHOTS_TEST7 = 10


def build_layers(gate_specs):
    return tuple(
        CircuitLayer(layer_index=index, gate_name=gate_name, qubits=tuple(qubits))
        for index, (gate_name, qubits) in enumerate(gate_specs)
    )


def build_circuit(gate_specs):
    return CircuitDescription(layers=build_layers(gate_specs))


def summarize_ultra_batch(batch):
    nontrivial_counts = []
    physical_counters = None
    toggling_counters = None
    xi_values = []
    for process in batch.processes:
        xi_values.append(process.hidden_memory.xi)
        nontrivial_counts.append(sum(1 for fault in process.block_faults if fault.physical_pauli_string is not None))
        if physical_counters is None:
            physical_counters = [Counter() for _ in process.block_faults]
            toggling_counters = [Counter() for _ in process.block_faults]
        for block_index, fault in enumerate(process.block_faults):
            physical_label = fault.physical_pauli_string if fault.physical_pauli_string is not None else 'I'
            toggling_label = fault.toggling_frame_pauli_string if fault.toggling_frame_pauli_string is not None else 'I'
            physical_counters[block_index][physical_label] += 1
            toggling_counters[block_index][toggling_label] += 1
    physical_per_block_frequencies = [
        {label: count / batch.num_shots for label, count in sorted(counter.items())}
        for counter in physical_counters
    ]
    toggling_per_block_frequencies = [
        {label: count / batch.num_shots for label, count in sorted(counter.items())}
        for counter in toggling_counters
    ]
    return {
        'num_shots': batch.num_shots,
        'mean_xi': float(np.mean(xi_values)),
        'mean_xi2': float(np.mean(np.square(xi_values))),
        'average_nontrivial_block_faults': float(np.mean(nontrivial_counts)),
        'physical_per_block_frequencies': physical_per_block_frequencies,
        'toggling_per_block_frequencies': toggling_per_block_frequencies,
        'per_block_frequencies': physical_per_block_frequencies,
    }


def ideal_gate_matrix(layer, num_qubits):
    local_single = {
        'I': np.array([[1.0, 0.0], [0.0, 1.0]], dtype=complex),
        'X': np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex),
        'Y': np.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=complex),
        'Z': np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex),
        'H': (1.0 / np.sqrt(2.0)) * np.array([[1.0, 1.0], [1.0, -1.0]], dtype=complex),
        'S': np.array([[1.0, 0.0], [0.0, 1.0j]], dtype=complex),
    }
    if layer.gate_name in local_single:
        target = layer.qubits[0]
        matrix = np.array([[1.0]], dtype=complex)
        for qubit in range(num_qubits):
            matrix = np.kron(matrix, local_single[layer.gate_name] if qubit == target else local_single['I'])
        return matrix
    if layer.gate_name == 'CNOT':
        control, target = layer.qubits
        dimension = 2 ** num_qubits
        matrix = np.zeros((dimension, dimension), dtype=complex)
        for basis_index in range(dimension):
            bits = [(basis_index >> (num_qubits - 1 - qubit)) & 1 for qubit in range(num_qubits)]
            if bits[control] == 1:
                bits[target] ^= 1
            output_index = 0
            for bit in bits:
                output_index = (output_index << 1) | bit
            matrix[output_index, basis_index] = 1.0
        return matrix
    raise ValueError(f'Unsupported gate for test helper: {layer.gate_name}')


def apply_ultra_process_to_density_matrix(initial_density_matrix, circuit, process):
    density_matrix = np.array(initial_density_matrix, dtype=complex, copy=True)
    num_qubits = int(np.log2(initial_density_matrix.shape[0]))
    blocks = build_fixed_window_blocks(circuit, window_size=WINDOW_SIZE)
    for block, fault in zip(blocks, process.block_faults):
        for layer_index in range(block.start_segment, block.end_segment + 1):
            layer = circuit.layers[layer_index]
            gate_matrix = ideal_gate_matrix(layer, num_qubits)
            density_matrix = gate_matrix @ density_matrix @ gate_matrix.conj().T
        if fault.physical_pauli_string is not None:
            pauli_matrix = pauli_string_to_matrix(fault.physical_pauli_string)
            density_matrix = pauli_matrix @ density_matrix @ pauli_matrix.conj().T
    return density_matrix


def average_density_matrix_over_ultra_batch(initial_density_matrix, circuit, batch):
    averaged = np.zeros_like(initial_density_matrix, dtype=complex)
    for process in batch.processes:
        averaged += apply_ultra_process_to_density_matrix(initial_density_matrix, circuit, process)
    return averaged / batch.num_shots


def pauli_expectation(density_matrix, pauli_string):
    pauli_matrix = pauli_string_to_matrix(pauli_string)
    return float(np.real_if_close(np.trace(density_matrix @ pauli_matrix)))



## Frame Mapping Micro-Check

Before the larger tests, this short example isolates the corrected simulator convention. The block Pauli distribution is computed in the toggling frame, then the sampled fault is conjugated into the physical inserted fault at the end of the block.


In [ ]:
frame_demo_circuit = build_circuit([('H', (0,))])
frame_demo_model = SharedQuasiStaticModel(
    num_qubits=1,
    sigma2=0.0,
    segment_durations=(0.25,),
)
frame_demo_process = build_blockwise_hidden_memory_process(
    frame_demo_model,
    frame_demo_circuit,
    window_size=1,
    memory=HiddenMemorySample(xi=np.pi),
    rng=np.random.default_rng(0),
)
frame_demo_fault = frame_demo_process.block_faults[0]

print('Frame demo block probabilities (toggling frame) =')
print({k: v for k, v in frame_demo_process.block_channels[0].probabilities.items() if v > 1e-12})
print('Sampled toggling-frame fault Q_b =', frame_demo_fault.toggling_frame_pauli_string)
print('Sampled physical inserted fault F_b =', frame_demo_fault.physical_pauli_string)
print('Compatibility alias fault.pauli_string =', frame_demo_fault.pauli_string)



## Test 1: One-Qubit Free Evolution

Target:

- `n=1`
- `L=8`
- all ideal gates `I`
- blockwise hidden-memory process with fixed `window_size = 2`
- report block-level sampled results with `num_shots = 100`

In [3]:
circuit_test1 = build_circuit([('I', (0,))] * 8)
model_test1 = SharedQuasiStaticModel(
    num_qubits=1,
    sigma2=0.125,
    segment_durations=(0.25,) * 8,
)
batch_test1 = sample_blockwise_hidden_memory_processes(
    model_test1,
    circuit_test1,
    window_size=WINDOW_SIZE,
    num_shots=NUM_SHOTS,
    rng=np.random.default_rng(101),
)
summary_test1 = summarize_ultra_batch(batch_test1)
first_process_test1 = batch_test1.processes[0]

print('Test 1 parameters:')
print('window_size =', WINDOW_SIZE)
print('num_shots =', summary_test1['num_shots'])
print('mean(xi^2) =', summary_test1['mean_xi2'])
print('average nontrivial block faults =', summary_test1['average_nontrivial_block_faults'])
print('block frequencies:')
for block_index, frequencies in enumerate(summary_test1['per_block_frequencies']):
    print(block_index, frequencies)
print('first process block-channel probabilities (block 0):')
print({k: v for k, v in first_process_test1.block_channels[0].probabilities.items() if v > 1e-12})


Test 1 parameters:
window_size = 2
num_shots = 100
mean(xi^2) = 0.1332378378685378
average nontrivial block faults = 0.1
block frequencies:
0 {'I': 0.97, 'Z': 0.03}
1 {'I': 0.96, 'Z': 0.04}
2 {'I': 0.99, 'Z': 0.01}
3 {'I': 0.98, 'Z': 0.02}
first process block-channel probabilities (block 0):
{'I': 0.9806159035148732, 'Z': 0.019384096485126918}


### Test 1 Interpretation

With `window_size = 2`, the one-qubit free case groups the 8 raw segments into 4 identical two-segment blocks.

The expected blockwise behavior is:

- each block accumulates a free rotation `e^{-i 2 xi Delta Z}` before projection,
- each projected block therefore becomes a nontrivial `Z`-dephasing channel,
- sampled nontrivial block faults should therefore occur at a visible but still sparse rate.

## Test 2: One-Qubit All-`X` Case

Target:

- `n=1`
- `L=8`
- all ideal gates `X`
- same blockwise process, same `window_size = 2`
- compare against Test 1

In [4]:
circuit_test2 = build_circuit([('X', (0,))] * 8)
model_test2 = SharedQuasiStaticModel(
    num_qubits=1,
    sigma2=0.125,
    segment_durations=(0.25,) * 8,
)
batch_test2 = sample_blockwise_hidden_memory_processes(
    model_test2,
    circuit_test2,
    window_size=WINDOW_SIZE,
    num_shots=NUM_SHOTS,
    rng=np.random.default_rng(102),
)
summary_test2 = summarize_ultra_batch(batch_test2)
first_process_test2 = batch_test2.processes[0]

print('Test 2 parameters:')
print('window_size =', WINDOW_SIZE)
print('num_shots =', summary_test2['num_shots'])
print('mean(xi^2) =', summary_test2['mean_xi2'])
print('average nontrivial block faults =', summary_test2['average_nontrivial_block_faults'])
print('block frequencies:')
for block_index, frequencies in enumerate(summary_test2['per_block_frequencies']):
    print(block_index, frequencies)
print('first process block-channel probabilities (block 0):')
print({k: v for k, v in first_process_test2.block_channels[0].probabilities.items() if v > 1e-12})


Test 2 parameters:
window_size = 2
num_shots = 100
mean(xi^2) = 0.1368438110308192
average nontrivial block faults = 0.0
block frequencies:
0 {'I': 1.0}
1 {'I': 1.0}
2 {'I': 1.0}
3 {'I': 1.0}
first process block-channel probabilities (block 0):
{'I': 1.0}


### Test 2 Interpretation

For the canonical sign-flip DD case, a two-segment block is the minimal cancellation block.

The expected blockwise behavior is:

- each `(+Z,-Z)` pair should cancel before projection,
- the projected two-segment block channel should therefore be identity,
- sampled nontrivial block faults should disappear or become numerically negligible.

## Test 3: Two-Qubit Free Shared Noise

Target:

- `n=2`
- free shared noise
- keep `window_size = 2`
- inspect representative block-level sampled statistics

In [5]:
circuit_test3 = build_circuit([('I', (0,)), ('I', (1,))] * 4)
model_test3 = SharedQuasiStaticModel(
    num_qubits=2,
    sigma2=0.125,
    segment_durations=(0.25,) * 8,
)
batch_test3 = sample_blockwise_hidden_memory_processes(
    model_test3,
    circuit_test3,
    window_size=WINDOW_SIZE,
    num_shots=NUM_SHOTS,
    rng=np.random.default_rng(103),
)
summary_test3 = summarize_ultra_batch(batch_test3)
first_process_test3 = batch_test3.processes[0]

print('Test 3 parameters:')
print('window_size =', WINDOW_SIZE)
print('num_shots =', summary_test3['num_shots'])
print('mean(xi^2) =', summary_test3['mean_xi2'])
print('average nontrivial block faults =', summary_test3['average_nontrivial_block_faults'])
print('block 0 frequencies =', summary_test3['per_block_frequencies'][0])
print('block 1 frequencies =', summary_test3['per_block_frequencies'][1])
print('first process block-channel probabilities (block 0):')
print({k: v for k, v in first_process_test3.block_channels[0].probabilities.items() if v > 1e-12})


Test 3 parameters:
window_size = 2
num_shots = 100
mean(xi^2) = 0.15021067852769765
average nontrivial block faults = 0.35
block 0 frequencies = {'I': 0.92, 'IZ': 0.05, 'ZI': 0.03}
block 1 frequencies = {'I': 0.93, 'IZ': 0.03, 'ZI': 0.04}
first process block-channel probabilities (block 0):
{'II': 0.9262848947915712, 'IZ': 0.03615206147703096, 'ZI': 0.03615206147703096, 'ZZ': 0.001410982254366909}


### Test 3 Interpretation

This test checks that the blockwise path still produces nontrivial correlated Pauli structure for a small two-qubit free case.

The point here is not DD, but to confirm that moving from segmentwise to blockwise projection does not destroy ordinary correlated-env behavior on a small instance.

## Test 4: Two-Qubit Alternating-`CNOT` Cycle

Target:

- `n=2`
- `L=8`
- ideal layers alternate `CNOT(0,1)` and `CNOT(1,0)`
- keep `window_size = 2`
- compare the computed block probabilities against the manuscript's analytic Type-I / Type-II formulas



In [6]:
circuit_test4 = build_circuit([
    ('CNOT', (0, 1)), ('CNOT', (1, 0)),
    ('CNOT', (0, 1)), ('CNOT', (1, 0)),
    ('CNOT', (0, 1)), ('CNOT', (1, 0)),
    ('CNOT', (0, 1)), ('CNOT', (1, 0)),
])
model_test4 = SharedQuasiStaticModel(
    num_qubits=2,
    sigma2=0.125,
    segment_durations=(0.25,) * 8,
)
batch_test4 = sample_blockwise_hidden_memory_processes(
    model_test4,
    circuit_test4,
    window_size=WINDOW_SIZE,
    num_shots=NUM_SHOTS,
    rng=np.random.default_rng(104),
)
summary_test4 = summarize_ultra_batch(batch_test4)
first_process_test4 = batch_test4.processes[0]
xi_test4 = first_process_test4.hidden_memory.xi
delta_test4 = 0.25
p_test4 = np.sin(xi_test4 * delta_test4) ** 2
P_test4 = np.sin(2.0 * xi_test4 * delta_test4) ** 2
analytic_type_i = {
    'II': (1.0 - P_test4) ** 2,
    'ZI': P_test4 * (1.0 - P_test4),
    'ZZ': P_test4 * (1.0 - P_test4),
    'IZ': P_test4 ** 2,
}
analytic_type_ii = {
    'ZI': p_test4 * (1.0 - p_test4),
    'ZZ': p_test4 * (1.0 - p_test4),
    'II': (1.0 - P_test4) * (1.0 - p_test4) ** 2 + P_test4 * p_test4 ** 2,
    'IZ': (1.0 - P_test4) * p_test4 ** 2 + P_test4 * (1.0 - p_test4) ** 2,
}

def dense(probabilities):
    labels = ('II', 'IZ', 'ZI', 'ZZ')
    return {label: float(probabilities.get(label, 0.0)) for label in labels}

actual_block0 = dense(first_process_test4.block_channels[0].probabilities)
actual_block1 = dense(first_process_test4.block_channels[1].probabilities)
actual_block2 = dense(first_process_test4.block_channels[2].probabilities)
actual_block3 = dense(first_process_test4.block_channels[3].probabilities)
max_diff_block0 = max(abs(actual_block0[label] - analytic_type_i[label]) for label in actual_block0)
max_diff_block1 = max(abs(actual_block1[label] - analytic_type_ii[label]) for label in actual_block1)
max_diff_block2 = max(abs(actual_block2[label] - analytic_type_ii[label]) for label in actual_block2)
max_diff_block3 = max(abs(actual_block3[label] - analytic_type_i[label]) for label in actual_block3)

print('Test 4 parameters:')
print('window_size =', WINDOW_SIZE)
print('num_shots =', summary_test4['num_shots'])
print('mean(xi^2) =', summary_test4['mean_xi2'])
print('average nontrivial block faults =', summary_test4['average_nontrivial_block_faults'])
print('physical block frequencies:')
for block_index, frequencies in enumerate(summary_test4['physical_per_block_frequencies']):
    print(block_index, frequencies)
print('')
print('first-process xi =', xi_test4)
print('analytic Type-I probabilities =', {k: v for k, v in analytic_type_i.items() if v > 1e-12})
print('analytic Type-II probabilities =', {k: v for k, v in analytic_type_ii.items() if v > 1e-12})
print('actual block 0 probabilities =', {k: v for k, v in actual_block0.items() if v > 1e-12})
print('actual block 1 probabilities =', {k: v for k, v in actual_block1.items() if v > 1e-12})
print('actual block 2 probabilities =', {k: v for k, v in actual_block2.items() if v > 1e-12})
print('actual block 3 probabilities =', {k: v for k, v in actual_block3.items() if v > 1e-12})
print('max abs diff block 0 vs Type-I =', max_diff_block0)
print('max abs diff block 1 vs Type-II =', max_diff_block1)
print('max abs diff block 2 vs Type-II =', max_diff_block2)
print('max abs diff block 3 vs Type-I =', max_diff_block3)



Test 4 parameters:
window_size = 2
num_shots = 100
mean(xi^2) = 0.12905811142455031
average nontrivial block faults = 0.13
block frequencies:
0 {'I': 0.94, 'IZ': 0.01, 'ZI': 0.05}
1 {'I': 0.93, 'IZ': 0.01, 'ZI': 0.06}
first process block-channel probabilities (block 0):
{'II': 0.9852798177982548, 'ZI': 0.009799299408410022, 'IZ': 0.0024604413966675697, 'ZZ': 0.0024604413966675697}


### Test 4 Interpretation

This test now matches the new analytic alternating-`CNOT` example in `lichen_manuscript.md`.

The expected `window_size = 2` structure is:

- blocks 0 and 3 are Type-I blocks,
- blocks 1 and 2 are Type-II blocks,
- the printed max-difference checks should stay at numerical-noise scale if the code matches the manuscript derivation.



## Test 5: Observable-Level Free Versus DD Comparison

Target:

- initial state `|+><+|`
- compare final averaged `⟨X⟩` for Test 1 and Test 2
- same `window_size = 2` and same `num_shots = 100`

In [7]:
rho_plus = 0.5 * np.array([[1.0, 1.0], [1.0, 1.0]], dtype=complex)

rho_out_test1 = average_density_matrix_over_ultra_batch(rho_plus, circuit_test1, batch_test1)
rho_out_test2 = average_density_matrix_over_ultra_batch(rho_plus, circuit_test2, batch_test2)

x_expectation_test1 = pauli_expectation(rho_out_test1, 'X')
x_expectation_test2 = pauli_expectation(rho_out_test2, 'X')

print('Test 5 output density matrices:')
print('Test 1 average output =')
print(rho_out_test1)
print('Test 2 average output =')
print(rho_out_test2)
print('')
print('Test 5 X expectations:')
print('Test 1: <X> =', x_expectation_test1)
print('Test 2: <X> =', x_expectation_test2)
print('Difference (Test 2 - Test 1) =', x_expectation_test2 - x_expectation_test1)


Test 5 output density matrices:
Test 1 average output =
[[0.5 +0.j 0.42+0.j]
 [0.42+0.j 0.5 +0.j]]
Test 2 average output =
[[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]

Test 5 X expectations:
Test 1: <X> = 0.84
Test 2: <X> = 1.0
Difference (Test 2 - Test 1) = 0.16000000000000003


### Test 5 Interpretation

This is the decisive observable-level check.

The helper above applies the sampled **physical inserted** block faults, not the raw toggling-frame block labels. That is the correct simulator-facing convention after the frame fix.

If the blockwise construction is doing what `lichen_manuscript.md` claims for the canonical sign-flip case, then Test 2 should preserve more `X` coherence than Test 1. With `window_size = 2`, that contrast should already appear because a two-segment block is the minimal DD cancellation block.


## Test 6: Medium Multi-Qubit Clifford Example

Target:

- `n=3`
- medium nontrivial Clifford circuit
- same `window_size = 2`
- inspect whether the blockwise path stays structurally coherent beyond the one-qubit sanity checks

In [8]:
circuit_test6 = build_circuit([
    ('H', (0,)),
    ('CNOT', (0, 1)),
    ('S', (2,)),
    ('CNOT', (1, 2)),
    ('H', (1,)),
    ('CNOT', (0, 2)),
])
model_test6 = SharedQuasiStaticModel(
    num_qubits=3,
    sigma2=0.125,
    segment_durations=(0.25,) * circuit_test6.circuit_depth,
)
batch_test6 = sample_blockwise_hidden_memory_processes(
    model_test6,
    circuit_test6,
    window_size=WINDOW_SIZE,
    num_shots=NUM_SHOTS,
    rng=np.random.default_rng(106),
)
summary_test6 = summarize_ultra_batch(batch_test6)
first_process_test6 = batch_test6.processes[0]

print('Test 6 parameters:')
print('window_size =', WINDOW_SIZE)
print('num_shots =', summary_test6['num_shots'])
print('mean(xi^2) =', summary_test6['mean_xi2'])
print('average nontrivial block faults =', summary_test6['average_nontrivial_block_faults'])
print('block frequencies:')
for block_index, frequencies in enumerate(summary_test6['per_block_frequencies']):
    print(block_index, frequencies)
print('first process block-channel probabilities (block 0):')
print({k: v for k, v in first_process_test6.block_channels[0].probabilities.items() if v > 1e-12})


Test 6 parameters:
window_size = 2
num_shots = 100
mean(xi^2) = 0.12327346320666235
average nontrivial block faults = 0.18
block frequencies:
0 {'I': 0.97, 'IIZ': 0.03}
1 {'I': 0.92, 'IIZ': 0.02, 'XXX': 0.01, 'YYI': 0.01, 'ZIZ': 0.01, 'ZZI': 0.03}
2 {'I': 0.93, 'IIZ': 0.01, 'XZI': 0.01, 'YYI': 0.01, 'YYZ': 0.01, 'ZXI': 0.03}
first process block-channel probabilities (block 0):
{'III': 0.9523594850899743, 'IIZ': 0.02356603254400604, 'IZI': 0.005819723919668983, 'XII': 0.005819723919668983, 'XXI': 0.005819723919668983, 'ZZI': 0.005819723919668983, 'IZZ': 0.00014400843949707984, 'XIZ': 0.00014400843949707984, 'XXZ': 0.00014400843949707984, 'ZZZ': 0.00014400843949707984, 'XZI': 3.556344744964402e-05, 'YYI': 3.556344744964402e-05, 'IXI': 3.556344744964401e-05, 'XYI': 3.556344744964401e-05, 'YZI': 3.556344744964401e-05, 'ZII': 3.556344744964401e-05, 'IXZ': 8.800136640589881e-07, 'XYZ': 8.800136640589881e-07, 'XZZ': 8.800136640589881e-07, 'YYZ': 8.800136640589881e-07, 'YZZ': 8.800136640589881

### Test 6 Interpretation

This medium case checks that the blockwise machinery works cleanly on a small multi-qubit circuit with mixed single- and two-qubit Clifford layers.

## Test 7: Big Clifford Circuit Sampled Check

Target:

- `n=8`
- bigger Clifford history
- same fixed `window_size = 2`
- run a real sampled `ultra` process, not only a structural partition check
- use a smaller shot count than Tests 1-6 because the exact blockwise path is still significantly more expensive than `window_size = 1` / `max`


In [9]:
circuit_test7 = build_circuit([
    ('H', (0,)), ('H', (2,)), ('H', (4,)), ('H', (6,)),
    ('CNOT', (0, 1)), ('CNOT', (2, 3)), ('CNOT', (4, 5)), ('CNOT', (6, 7)),
    ('S', (1,)), ('S', (3,)), ('S', (5,)), ('S', (7,)),
    ('CNOT', (1, 2)), ('CNOT', (3, 4)), ('CNOT', (5, 6)),
    ('H', (1,)), ('H', (3,)), ('H', (5,)),
    ('CNOT', (0, 2)), ('CNOT', (3, 5)), ('CNOT', (4, 6)),
    ('S', (0,)), ('S', (2,)), ('S', (4,)), ('S', (6,)),
    ('CNOT', (2, 4)), ('CNOT', (1, 3)), ('CNOT', (5, 7)),
])
model_test7 = SharedQuasiStaticModel(
    num_qubits=8,
    sigma2=0.125,
    segment_durations=(0.25,) * circuit_test7.circuit_depth,
)
blocks_test7 = build_fixed_window_blocks(circuit_test7, window_size=WINDOW_SIZE)
batch_test7 = sample_blockwise_hidden_memory_processes(
    model_test7,
    circuit_test7,
    window_size=WINDOW_SIZE,
    num_shots=NUM_SHOTS_TEST7,
    rng=np.random.default_rng(107),
)
summary_test7 = summarize_ultra_batch(batch_test7)
first_process_test7 = batch_test7.processes[0]

print('Test 7 parameters:')
print('window_size =', WINDOW_SIZE)
print('num_shots =', summary_test7['num_shots'])
print('num_segments =', circuit_test7.circuit_depth)
print('num_blocks =', len(blocks_test7))
print('mean(xi^2) =', summary_test7['mean_xi2'])
print('average nontrivial block faults =', summary_test7['average_nontrivial_block_faults'])
print('first six blocks:')
for block in blocks_test7[:6]:
    print(block)
print('block 0 frequencies =', summary_test7['per_block_frequencies'][0])
print('block 1 frequencies =', summary_test7['per_block_frequencies'][1])
print('first process block-channel probabilities (block 0):')
print({k: v for k, v in first_process_test7.block_channels[0].probabilities.items() if v > 1e-12})
print('block 0 discarded weight =', first_process_test7.block_channels[0].discarded_weight)
print('')
print('Test 7 note: `window_size = 2` is physically better than `window_size = 1`, but it costs noticeably more resources because projection is delayed until after coherent block propagation.')


Test 7 parameters:
window_size = 2
num_shots = 10
num_segments = 28
num_blocks = 14
mean(xi^2) = 0.09333573569981295
average nontrivial block faults = 1.8
first six blocks:
BlockDescriptor(block_index=0, start_segment=0, end_segment=1)
BlockDescriptor(block_index=1, start_segment=2, end_segment=3)
BlockDescriptor(block_index=2, start_segment=4, end_segment=5)
BlockDescriptor(block_index=3, start_segment=6, end_segment=7)
BlockDescriptor(block_index=4, start_segment=8, end_segment=9)
BlockDescriptor(block_index=5, start_segment=10, end_segment=11)
block 0 frequencies = {'I': 0.8, 'IIIIIIZI': 0.1, 'IIIIIZII': 0.1}
block 1 frequencies = {'I': 0.9, 'XZIIIIII': 0.1}
first process block-channel probabilities (block 0):
{'IIIIIIII': 0.9667561983857473, 'IIIIIIIZ': 0.0043680017752844745, 'IIIIIIZI': 0.004368001775284473, 'IIIIIZII': 0.004368001775284473, 'IIIIZIII': 0.004368001775284469, 'IIIZIIII': 0.004368001775284466, 'IZIIIIII': 0.0043680017752844545, 'XIIIIIII': 0.004368001775284436, 'IIX

### Test 7 Interpretation

This large case is now a real sampled `ultra` simulation, not only a structural placeholder.

The important tradeoff is explicit here:

- `window_size = 2` preserves more DD-like block physics than `window_size = 1` / `max`,
- but it costs noticeably more resources because projection is delayed until after coherent block propagation.

So Test 7 now verifies both:

- the big 8-qubit circuit can be sampled by the current exact sparse-convolution `ultra` path,
- and the remaining limitation is batch cost, not outright infeasibility.


## Test 8: Ultra Truncation Probe

Target:

- reuse the large `n=8` Test 7 circuit
- compare the exact exported block distributions against a truncated export
- inspect whether truncation mainly reduces runtime, exported support size, or both


In [10]:
truncation_config = SparseExportConfig(
    top_k_non_identity=16,
    probability_threshold=1e-8,
    max_pauli_weight=2,
)

start_exact = time.perf_counter()
batch_test8_exact = sample_blockwise_hidden_memory_processes(
    model_test7,
    circuit_test7,
    window_size=WINDOW_SIZE,
    num_shots=1,
    rng=np.random.default_rng(208),
)
elapsed_exact = time.perf_counter() - start_exact

start_trunc = time.perf_counter()
batch_test8_trunc = sample_blockwise_hidden_memory_processes(
    model_test7,
    circuit_test7,
    window_size=WINDOW_SIZE,
    num_shots=1,
    export_config=truncation_config,
    rng=np.random.default_rng(208),
)
elapsed_trunc = time.perf_counter() - start_trunc

exact_block0 = batch_test8_exact.processes[0].block_channels[0]
trunc_block0 = batch_test8_trunc.processes[0].block_channels[0]

print('Test 8 parameters:')
print('window_size =', WINDOW_SIZE)
print('truncation_config =', truncation_config)
print('')
print('timing comparison for one sampled 8-qubit shot:')
print('exact export elapsed =', elapsed_exact)
print('truncated export elapsed =', elapsed_trunc)
print('')
print('block 0 support comparison:')
print('exact kept support size =', len(exact_block0.probabilities))
print('truncated kept support size =', len(trunc_block0.probabilities))
print('truncated kept probability mass =', trunc_block0.kept_probability_mass)
print('truncated discarded weight =', trunc_block0.discarded_weight)
print('')
print('top exact block-0 probabilities:')
print(dict(list(sorted(exact_block0.probabilities.items(), key=lambda kv: (-kv[1], kv[0]))[:20])))
print('')
print('truncated block-0 probabilities:')
print(trunc_block0.probabilities)


Test 8 parameters:
window_size = 2
truncation_config = SparseExportConfig(top_k_non_identity=16, probability_threshold=1e-08, max_pauli_weight=2, normalization_mode='renormalize_kept')

timing comparison for one sampled 8-qubit shot:
exact export elapsed = 3.6052614709999986
truncated export elapsed = 3.752033429000001

block 0 support comparison:
exact kept support size = 347
truncated kept support size = 17
truncated kept probability mass = 0.9998994906718782
truncated discarded weight = 0.00010050932812177837

top exact block-0 probabilities:
{'IIIIIIII': 0.982133732643094, 'IIIIIIIZ': 0.0023636478360209226, 'IIIIIIZI': 0.0023636478360209217, 'IIIIIZII': 0.0023636478360209213, 'IIIIZIII': 0.0023636478360209204, 'IIIZIIII': 0.0023636478360209174, 'IZIIIIII': 0.0023636478360209118, 'XIIIIIII': 0.0023636478360209018, 'IIXIIIII': 0.0005902019689108034, 'IIZIIIII': 0.0005902019689108034, 'XZIIIIII': 5.688462687959088e-06, 'XIIZIIII': 5.6884626879590815e-06, 'XIIIZIII': 5.688462687959076e

### Test 8 Interpretation

This probe is intended to separate two different effects:

- reducing the **exported block alphabet**,
- reducing the **exact computation cost** of the blockwise `ultra` path.

If truncation strongly reduces support size but only weakly changes runtime, then the remaining cost is still dominated by exact block probability construction, not by storing or sampling the exported support.
